# Identifying good reviews

Let's create a model to identify helpful reviews that people vote as helpful. They could be low rating reviews,

## Load Data

In [1]:
# enable autoreload
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
from evalforge.utils import *

dataset_path = Path("data/clothes_review_filtered.jsonl")
data = load_jsonl(dataset_path)

print(f"Number of examples: {len(data)}")
print(f"Number of reviews: {sum(len(example['reviews']) for example in data)}")
print("-"*100)
pprint(data[0])

Number of examples: 31650
Number of reviews: 42148
----------------------------------------------------------------------------------------------------
{   'asin': '9792252916',
    'author': None,
    'average_rating': 4.4,
    'bought_together': None,
    'categories': ['Clothing, Shoes & Jewelry', 'Men', 'Watches', 'Wrist Watches'],
    'description': [   'A tried and true style that always remains in fashion. With its daily '
                       'alarm, hourly time signal and auto calendar, you’ll never need to worry '
                       'about missing an appointment again. Black casual classic watch with a '
                       'resin Band.',
                       'With its iconic digital design and host of features, the Classic Watch '
                       "#59-1V from Casio offers micro-light illumination that's great for casual "
                       'day or evening wear. This compact and sporty timepiece also includes a '
                       '1/100-second sto

In [3]:
item = data[0]
reviews = item["reviews"]

## LLM

In [4]:
import weave
weave.init("amazon_fashion")


Logged in as Weights & Biases user: capecape.
View Weave data at https://wandb.ai/capecape/amazon_fashion/weave


In [5]:
import instructor
# from litellm import acompletion

import openai


# llm_client = instructor.from_litellm(acompletion)
oai_client = openai.AsyncOpenAI()
instructor_client = instructor.from_openai(oai_client)


In [6]:
system_prompt = """# Constructing a LLM Judge Benchmark

## The Benchmark
I am trying to build a benchmark for an LLM judge. This benchmark requires positive and negative labels for a given AI-generated output. I have a dataset of Amazon product descriptions and customer reviews which I think I can use. I will construct a benchmark dataset of product descriptions and an associated good / bad ground truth set of labels for the quality of the description text.
I will then pass the product descriptions to a LLM Judge and ask it to rate the descriptions. Then I will compare the Judge's labels with my ground truth labels in order to understand how aligned.

## Using Reviews As A Proxy signal For Description Text Quality
I want to use the product descriptions as a proxy for AI-generated output and use the feedback provided in the customer review texts as a proxy signal into the quality of the description text.

## Product Description Constraints
However I do not have the actual products in my hand so I am constrained to only assessing the quality of the description text, without knowing if the text matches the actual product in reality. Therefore I am just assessing whether the description text is clear and well-written or whether it is missing information or badly formatted or uses bad english, bad grammar etc. I cannot know if the description text is misleading as I cannot compare to the actual product in reality.

## Scoring Rubric
Given the customer reviews of descriptions please judge the following

- `is_useful`: Is the review useful in identifying good or bad descriptions.

A useful review is one where the review text provides a clear signal about the quality of the product description. The sentiment of the review doesn't matter - it could be a positive or negative review.

For example a useful review could be one that indicates a bad description, for instance if the product description is missing key information or is badly worded, uses poor grammar or is poorly styled or formatted. On the other hand a useful review could indicate a good product description, for example if the reviewer finds that the product description is accurate and helpful. For a review to be useful for this task, the review should explicitly mention the product description in some way.

A review is not useful for this task if:

- it doesn't mention the product description
- it says the description is misleading, as we cannot know whether or not the description is misleading without also having the product in our hands to compare it to
- it only makes indirect or implicit references to the description being good or bad

## Examples
{examples}
"""


In [7]:
examples = """
Examples of reviews that are useful and not useful:

### useful examples

- I ordered 3 of these shirts and am mostly satisfied. The shirts are great for exercising and the sizing is pretty accurate. My only complaint is that I think I'd rather a cotton/polyester blend instead of 100% polyester. While the polyester keeps you dry, the shirt is a little light and slinky and I'd rather some of the weight that cotton would bring. Other that that, these shirts are well made and are as described
- Hoop is just as described, fit my 7 year old perfectly. It did have a smell when we first opened it but I didn't notice it nearly as much once it aired out. My daughter loved what it did to her dress. Would love to post picts just don't know how!
- This purse was exactly as described and pictured. I was nervous since I have always carried huge shoulder bags so this was a down size for me. I have been wanting a cross body purse so I don't have to do everything with one hand It had enough room to support my kindle voyage, a diary, pens, bill fold, portable charger for phone, my huge keychain with my thousand of tags, phone, and chap stick. Wonderful and will be ordering from this seller again
- Great boots. I checked on UGG website and they are authentic. I believe they are true to size. I would advise against buying a size smaller, as suggested by the description. These are my first UGG boots. So far I am very happy with them. These are my first UGG boots. So far I am very happy with them
- Everything I hoped it would be (except the size)<br />Love the fabric on my skin, soft & lightweight as described
- I don't understand how this has so many stars from people, these sandals run extremely small. The title says men's 34, but should say unisex and have a better description for sizes.
- I ordered 2 of these shirts ant they are great, but it didn't say that is is 100% polyester.
- The watch didn't include the wristband, I would expect a full functioning watch!


### non useful examples
- I marked these as &#34;fit as expected&#34;, but not at first. This was the third attempt to get a fitting pair. Ecco is notorious for mislabeling the US equivalent size, and these were no exception
- I recieved an empty freakin G-shock box and knew it from the time i picked it up it was so under weight!!!!! I want my watch or a refund ASAP!
- I purchased 1 previously & luv it! I had  another plain anklet that just broke so I said, hey, why not have matching ones!!! It goes with everything & I luv the way it sparkles!! Looks expensive!
- So the small is for anyone who is an adult female size 5 or smaller.  Just FYI.  Medium is recommended for female US 5-9.  I would check out the socks on another site for size features before ordering to ensure you're getting the right size for your feet. I ordered blindly for a gift; fortunately, the person is small and the small socks fit her well, despite her shoe size being a US 7.
- The gold hoop earrings were a nice shiny gold and the shape and size were as described. However, it is very difficult to close the clasp because it is very thin and flimsy.. it feels that if I am not very careful, the loops will bend. Somewhat disappointed in this regard, but not surprised given the price. Also, someone else mentioned in their review that they were difficult to clasp.
"""

prompt_template = """
The item to review is:

## Item Name
{title}

## Review

Title: {review_title}

{review_content}

Score if this review `is_useful`.
"""

In [8]:
from typing import Literal
from pydantic import BaseModel, Field

class ReviewEvaluation(BaseModel):
    note: str = Field(description="Reason for the evaluation")
    is_useful: bool = Field(description="Is the review useful in identifying good or bad descriptions?")

In [9]:
def format_example(item: dict, review: dict):
    return prompt_template.format(
        title=item["title"],
        review_rating=review["rating"],
        review_title=review["title"],
        review_content=review["text"],
    )

In [10]:
print(format_example(item, reviews[0]))


The item to review is:

## Item Name
Casio Men's W59-1V Classic Black Digital Watch

## Review

Title: Waterproof F-91W

This watch is similar in looks and functionality as the famous F-91W, but might be slightly more durable.  The main difference is that this watch (the W-59) is 50M water resistant while the F-91W is technically just splash resistant (though in practice it can often handle more).  The module is slightly different from the F-91W, but it has all the same features including the sad back light.  The case is about 1mm shorter and about 1/4mm thicker.  The strap is similar in style, but about 0.25mm thicker as well.

Score if this review `is_useful`.



## Helpful reviews

Let's look at the helpful reviews

In [11]:
HELPFUL_VOTE_THRESHOLD = 5

In [12]:
helpful_reviews = []

for item in data:
    for review in item["reviews"]:
        if review["helpful_vote"] > HELPFUL_VOTE_THRESHOLD:
            item_without_reviews = {k: v for k, v in item.items() if k != "reviews"}
            helpful_reviews.append({"item":item_without_reviews, "review": review})

print(f"Number of helpful reviews: {len(helpful_reviews)}")


Number of helpful reviews: 27279


In [24]:
import asyncio
from tqdm.asyncio import tqdm

# MODEL_NAME = "o1-mini"
MODEL_NAME= "gpt-4o-mini"

async def async_map(func, items, max_concurrent=5, desc="Processing"):
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def wrapped_func(**item):
        async with semaphore:
            return await func(**item)
    
    tasks = [wrapped_func(**item) for item in items]
    return await tqdm.gather(*tasks, desc=desc)

@weave.op
async def extract_review_evaluation(review_evaluation: str) -> ReviewEvaluation:
    """Extract the review evaluation from the LLM response"""
    review_evaluation = await instructor_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": review_evaluation}
        ],
        response_model=ReviewEvaluation,
    )
    return review_evaluation.model_dump()

@weave.op
async def call_openai(prompt: str, max_completion_tokens: int = 8000) -> dict:
    if "o1" in MODEL_NAME:
        o1_prompt = system_prompt.format(examples=examples) + prompt # no system prompt for o1
        out = await oai_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": o1_prompt}],
            max_completion_tokens=max_completion_tokens,
        )
    else:        
        out = await oai_client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt.format(examples=examples)},
                {"role": "user", "content": prompt}],
            max_completion_tokens=max_completion_tokens,
        )
    review_evaluation = await extract_review_evaluation(out.choices[0].message.content)
    return {"review_evaluation": review_evaluation}

@weave.op
async def evaluate_review(item, review) -> dict:
    """Evaluate a single review using the LLM"""
    o1_prompt = format_example(item, review)
    out =  await call_openai(o1_prompt)
    return {"item": item, "review": review, **out}

@weave.op
async def evaluate_reviews(reviews: list[dict], max_concurrent: int = 25) -> list[dict]:
    """Evaluate a list of reviews"""
    with weave.attributes({"helpful_threshold": HELPFUL_VOTE_THRESHOLD}):
        return await async_map(evaluate_review, reviews, max_concurrent=max_concurrent, desc="Processing reviews")


## Dataset

In [14]:
annotations = await evaluate_reviews(helpful_reviews[:100], max_concurrent=25)
print(len(annotations))

Processing reviews: 100%|██████████| 100/100 [00:32<00:00,  3.11it/s]

🍩 https://wandb.ai/capecape/amazon_fashion/r/call/019301dc-53ec-72f0-9dac-947fc511451c
100


In [15]:
# out_file = "data/helpful_reviews_annotations_500_o1_useful.jsonl"
# save_jsonl(annotations, out_file)


## Analysis

In [16]:
from evalforge.utils import *

In [17]:
# annotations = load_jsonl(out_file)


In [18]:
useful_reviews = [d for d in annotations if d["review_evaluation"]["is_useful"]]
not_useful_reviews = [d for d in annotations if not d["review_evaluation"]["is_useful"]]

# print counts
print(f"Number of useful reviews: {len(useful_reviews)}")
print(f"Number of not useful reviews: {len(not_useful_reviews)}")


Number of useful reviews: 30
Number of not useful reviews: 70


In [19]:
for useful_review in useful_reviews:
    pprint({"review":useful_review["review"]["text"], **useful_review["review_evaluation"]})
    print("-"*100)

{   'is_useful': True,
    'note': 'The review explicitly mentions the product description, stating that "This watch does '
            'everything as described in the description." It provides clear feedback on the '
            'accuracy and completeness of the description, indicating that the description helped '
            'the user make an informed purchase. The review discusses specific aspects related to '
            "the description, such as the watch's functionality, size, and water resistance, which "
            'are directly tied to the quality of the product description.',
    'review': 'I bought this watch for me to know the time. I used to watch the time from my phone '
              'but that doesnt work for me because smartphones are not made especially for telling '
              'time. Everytime i watch the time from my smartphone it takes me minutes as I go to '
              'chat, games, facebook, twitter and so on.<br />I decide to leave my smartphone away '
  

In [22]:
wds = weave.Dataset(name="helpful_reviews_100", rows=[{"review":annotation["review"]["text"], **annotation["review_evaluation"]} for annotation in annotations])

In [23]:
weave.publish(wds)

📦 Published to https://wandb.ai/capecape/amazon_fashion/weave/objects/helpful_reviews_100/versions/nFG1FcXFUiGFJLGwq6S86FiTFLOyKP50c7VIiurX9uI


ObjectRef(entity='capecape', project='amazon_fashion', name='helpful_reviews_100', _digest='nFG1FcXFUiGFJLGwq6S86FiTFLOyKP50c7VIiurX9uI', _extra=())

save only good bad

In [ ]:
good_bad = good_descriptions + bad_descriptions
save_jsonl(good_bad, "data/helpful_reviews_annotations_1000_o1_mini_good_bad.jsonl")

In [35]:
# let's print the first 10 reviews for bad and good descriptions
for d in bad_descriptions:
    for line in d["review"]["text"].split("."):
        print(line)
    # print("-"*100)
    # for line in d["review_evaluation"]["note"].split("."):
    #     print(line)
    print("="*100)


I really like this ring and am very happy with it
  I don't wear it every day as I have quite a few rings that I switch off and on every day
  The diamonds were smaller than I thought they'd be, but the ring is beautiful
  You can wear this ring with another ring on the same finger if you prefer
  It also goes nicely wearing it with a wedding band or an anniversary diamond ring
  I recommend it for anyone wanting an affordable, very pretty yet simple ring
<br /><br />I am now editing this "great" review seven (7) months later
  I, too, (I should have paid more attention to the reviews), have lost the second to last small ruby
  I was told by the seller that basically unless a stone fell out in the first (30) days, I'm s*** out of luck
  You have to take it to a jeweler and have them repair it
  I saw reviews on here b/4 I bought the ring and should have listened to them
<br /><br />Beware that if you buy this ring, the ruby stones are known to fall out

Never had a nice watch before so

In [45]:
good_descriptions[0]

{'item': {'parent_asin': 'B0009G2LJ8',
  'main_category': 'AMAZON FASHION',
  'title': 'NOVICA Handcrafted Purple Batik Robe for Women - Front-Wrap Style, Wide Sleeves, and Matching Belt - Lightweight Robes for Women - 51" Long, Seaside Blue\'',
  'description': [],
  'average_rating': 4.1,
  'rating_number': 22,
  'asin': 'B0009G2LJ8',
  'features': ['Rayon',
   'Tie closure',
   'Hand Wash Only',
   'Authentic: an original NOVICA artisan handcrafted fair trade product.',
   'Certified: comes with an official NOVICA Story Card certifying quality & authenticity.',
   'NOVICA works with Desak Nyoman Parwati to craft this item.',
   'Exceptional Quality: crafted with care to be treasured as a keepsake for many years to come.',
   'Product info: Rayon,'],
  'price': '77.99',
  'images': {'hi_res': ['https://m.media-amazon.com/images/I/81Em6kNzMGL._AC_UL1500_.jpg',
    'https://m.media-amazon.com/images/I/81BEAet6VtL._AC_UL1500_.jpg',
    'https://m.media-amazon.com/images/I/61QuAphTQAL._A